# Лабораторная работа № 1
## Распознавание персональных данных

В работе распознаются 8 атрибутов ПДн:

- `NAME` — имя;
- `SURNAME` — фамилия;
- `EMAIL` — электронная почта;
- `INN` — ИНН;
- `PHONE` — номер телефона;
- `BANK_CARD` — номер банковской карты;
- `SNILS` — СНИЛС;
- `PASSPORT` — серия и номер паспорта.

Имена и фамилии распознаёт модель `DeepPavlov/rubert-base-cased`, обученная на указанном в задании наборе `factRuEval-2016`. Структурированные данные распознаются регулярными выражениями. Для ИНН, банковской карты и СНИЛС проверяются контрольные разряды.

> Ноутбук рассчитан на Google Colab. Перед запуском выберите: **Среда выполнения → Сменить среду выполнения → T4 GPU**. Выполняйте ячейки сверху вниз.

## 1. Установка библиотек в Google Colab

Эту ячейку нужно выполнить первой после открытия ноутбука в Google Colab. Команда `%pip` устанавливает пакеты именно в то окружение Python, в котором работает текущий ноутбук. Устанавливаются только реально используемые библиотеки:

- `corus` читает factRuEval;
- `transformers` и `torch` нужны для RuBERT;
- `datasets` хранит обучающую выборку;
- `seqeval` считает NER-метрики;
- `Faker` создаёт тестовые структурированные данные;
- `numpy` выполняет операции с массивами и выбирает наиболее вероятные метки;
- `tqdm` показывает индикаторы выполнения;
- `legacy-cgi` обеспечивает совместимость библиотеки `corus` с Python 3.13 и новее.

`seqeval` устанавливается отдельной командой через современный механизм сборки PEP 517. Это предотвращает ошибку `python setup.py egg_info`, которая возникает в некоторых средах Colab. `legacy-cgi` устанавливается только тогда, когда стандартного модуля `cgi` уже нет. `tokenizers` и другие внутренние зависимости будут установлены автоматически вместе с `transformers`.

In [1]:
# Выполнить первым в Google Colab. Повторный запуск безопасен.
%pip install -q -U setuptools wheel setuptools-scm
%pip install -q --use-pep517 seqeval==1.2.2
%pip install -q corus "transformers[torch]" datasets accelerate Faker numpy tqdm

# В Python 3.13 удалён модуль cgi, который нужен старой версии corus.
import importlib.util
import subprocess
import sys

if importlib.util.find_spec('cgi') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'legacy-cgi'])

print('Все библиотеки установлены.')

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: C:\Users\Admin\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: C:\Users\Admin\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Все библиотеки установлены.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: C:\Users\Admin\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


## 2. Загрузка factRuEval-2016

Архив скачивается из официального репозитория задания и распаковывается автоматически. Повторно он не скачивается.

In [2]:
from pathlib import Path
import urllib.request
import zipfile

DATASET_DIR = Path('factRuEval-2016-master')
ARCHIVE_PATH = Path('factRuEval-2016-master.zip')
DATASET_URL = 'https://github.com/dialogue-evaluation/factRuEval-2016/archive/master.zip'

if not DATASET_DIR.exists():
    urllib.request.urlretrieve(DATASET_URL, ARCHIVE_PATH)
    with zipfile.ZipFile(ARCHIVE_PATH) as archive:
        archive.extractall('.')
    ARCHIVE_PATH.unlink()

print('Набор данных:', DATASET_DIR.resolve())

Набор данных: C:\Users\Admin\programms\mei-czti\lab-1\factRuEval-2016-master


## 3. Загрузка данных и модели-токенизатора

`load_factru` читает исходные тексты и размеченные сущности. Для Windows явно задаётся UTF-8, потому что старая версия `corus` открывает основной текст в системной кодировке. Из всех типов factRuEval оставляем только требуемые заданием `NAME` и `SURNAME`.

BIO-схема:

- `B-NAME` — начало имени;
- `I-NAME` — продолжение имени;
- `B-SURNAME` и `I-SURNAME` — фамилия;
- `O` — токен не является нужной сущностью.

In [3]:
from collections import Counter
import numpy as np
import torch
from corus import load_factru
import corus.sources.factru as factru_source
from datasets import Dataset, DatasetDict
from tqdm import tqdm
from transformers import AutoTokenizer

MODEL_NAME = 'DeepPavlov/rubert-base-cased'
MAX_TOKEN_LENGTH = 512
TARGET_TYPES = {'NAME', 'SURNAME'}
LABEL_NAMES = ['O', 'B-NAME', 'I-NAME', 'B-SURNAME', 'I-SURNAME']

id2label = {index: label for index, label in enumerate(LABEL_NAMES)}
label2id = {label: index for index, label in id2label.items()}

# Исправление чтения factRuEval в Windows: все исходные файлы имеют UTF-8.
factru_source.load_text = lambda path: Path(path).read_text(encoding='utf-8')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
records = list(load_factru(str(DATASET_DIR)))

available_types = Counter(
    span.type.upper()
    for record in records
    for obj in record.objects
    for span in obj.spans
)
print('Документов:', len(records))
print('Размеченных имён:', available_types['NAME'])
print('Размеченных фамилий:', available_types['SURNAME'])
print('Метки модели:', label2id)

C:\Users\Admin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Документов: 254
Размеченных имён: 1343
Размеченных фамилий: 1966
Метки модели: {'O': 0, 'B-NAME': 1, 'I-NAME': 2, 'B-SURNAME': 3, 'I-SURNAME': 4}


## 4. Преобразование factRuEval в данные для token classification

RuBERT разбивает слова на субтокены WordPiece. Метка ставится только первому субтокену слова. Остальные субтокены и служебные токены получают `-100`, поэтому PyTorch не учитывает их при вычислении ошибки.

Для каждого токена проверяется пересечение его символьных координат со span-разметкой factRuEval.

In [4]:
def find_person_span(record, token_start, token_end):
    for obj in record.objects:
        for span in obj.spans:
            span_type = span.type.upper()
            overlaps = token_start < span.stop and span.start < token_end
            if span_type in TARGET_TYPES and overlaps:
                return span_type, span.start, span.stop
    return None

def records_to_dataset(selected_records, description):
    features = {
        'input_ids': [],
        'token_type_ids': [],
        'attention_mask': [],
        'labels': [],
    }

    for record in tqdm(selected_records, desc=description):
        encoded = tokenizer(
            record.text,
            truncation=True,
            max_length=MAX_TOKEN_LENGTH,
            return_overflowing_tokens=True,
        )

        for chunk_index, encoding in enumerate(encoded.encodings):
            labels = []
            previous_word_id = None
            previous_span = None
            for word_id, (start, end) in zip(encoding.word_ids, encoding.offsets):
                if word_id is None or word_id == previous_word_id:
                    labels.append(-100)
                else:
                    current_span = find_person_span(record, start, end)
                    if current_span is None:
                        labels.append(label2id['O'])
                        previous_span = None
                    else:
                        span_type = current_span[0]
                        prefix = 'I-' if current_span == previous_span else 'B-'
                        labels.append(label2id[prefix + span_type])
                        previous_span = current_span
                previous_word_id = word_id

            features['input_ids'].append(encoded['input_ids'][chunk_index])
            features['token_type_ids'].append(encoded['token_type_ids'][chunk_index])
            features['attention_mask'].append(encoded['attention_mask'][chunk_index])
            features['labels'].append(labels)

    return Dataset.from_dict(features)

## 5. Разделение на обучающую, проверочную и тестовую выборки

Сначала целые документы делятся воспроизводимо с `seed=42`, и только затем каждый документ разбивается на блоки до 512 токенов. Поэтому части одного документа не попадают одновременно в обучение и тест, а конец длинного документа не теряется.

Распределение:

- 80% — обучение;
- 10% — проверка после каждой эпохи;
- 10% — окончательный тест, который модель не видит при обучении.

In [5]:
from random import Random

shuffled_records = list(records)
Random(42).shuffle(shuffled_records)
train_end = int(len(shuffled_records) * 0.8)
valid_end = int(len(shuffled_records) * 0.9)

datasets = DatasetDict({
    'train': records_to_dataset(shuffled_records[:train_end], 'train'),
    'valid': records_to_dataset(shuffled_records[train_end:valid_end], 'valid'),
    'test': records_to_dataset(shuffled_records[valid_end:], 'test'),
})
print(datasets)

train:   0%|          | 0/203 [00:00<?, ?it/s]

train:  13%|█▎        | 27/203 [00:00<00:00, 262.63it/s]

train:  33%|███▎      | 68/203 [00:00<00:00, 346.41it/s]

train:  51%|█████     | 103/203 [00:00<00:00, 174.79it/s]

train:  63%|██████▎   | 127/203 [00:00<00:00, 146.25it/s]

...

valid: 100%|██████████| 25/25 [00:00<00:00, 249.92it/s]

test:   0%|          | 0/26 [00:00<?, ?it/s]

test:  92%|█████████▏| 24/26 [00:00<00:00, 239.52it/s]

test: 100%|██████████| 26/26 [00:00<00:00, 238.38it/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 265
    })
    valid: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 32
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 33
    })
})


## 6. Метрики precision, recall и F1

`seqeval` сравнивает целые BIO-сущности, а не только отдельные символы:

- **precision = TP / (TP + FP)** — доля правильных сущностей среди найденных;
- **recall = TP / (TP + FN)** — доля найденных сущностей среди существующих;
- **F1 = 2 · precision · recall / (precision + recall)** — баланс precision и recall.

In [6]:
from seqeval.metrics import classification_report
from seqeval.scheme import IOB2

def sequences_from_predictions(predictions, labels):
    predicted_ids = np.argmax(predictions, axis=-1)
    true_sequences = []
    predicted_sequences = []
    for predicted_row, label_row in zip(predicted_ids, labels):
        true_sequences.append([
            id2label[int(label)]
            for predicted, label in zip(predicted_row, label_row)
            if label != -100
        ])
        predicted_sequences.append([
            id2label[int(predicted)]
            for predicted, label in zip(predicted_row, label_row)
            if label != -100
        ])
    return true_sequences, predicted_sequences

def compute_metrics(evaluation_prediction):
    true_sequences, predicted_sequences = sequences_from_predictions(
        evaluation_prediction.predictions, evaluation_prediction.label_ids
    )
    report = classification_report(
        true_sequences,
        predicted_sequences,
        scheme=IOB2,
        output_dict=True,
        zero_division=0,
    )
    weighted = report['weighted avg']
    return {
        'precision': weighted['precision'],
        'recall': weighted['recall'],
        'f1': weighted['f1-score'],
    }

## 7. Обучение RuBERT

Предобученная русскоязычная модель получает новую голову классификации на 5 BIO-меток. Выполняются 3 эпохи; после каждой считается F1 на `valid`. В конце автоматически восстанавливается модель с лучшим F1. Если папка `res_model_base` уже существует, готовая модель загружается без повторного обучения.

На T4 обучение обычно занимает несколько минут. Если памяти GPU недостаточно, уменьшите `per_device_train_batch_size` с 8 до 4.

In [7]:
import os
from transformers import (
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
    set_seed,
)
from transformers.trainer_utils import get_last_checkpoint

os.environ['WANDB_DISABLED'] = 'true'
set_seed(42)

SAVED_MODEL = Path('res_model_base')
model_source = str(SAVED_MODEL) if (SAVED_MODEL / 'config.json').exists() else MODEL_NAME
model = AutoModelForTokenClassification.from_pretrained(
    model_source,
    num_labels=len(LABEL_NAMES),
    id2label=id2label,
    label2id=label2id,
)

training_args = TrainingArguments(
    output_dir='train_result',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    logging_steps=20,
    report_to='none',
    fp16=torch.cuda.is_available(),
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=datasets['train'],
    eval_dataset=datasets['valid'],
    processing_class=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
)

if model_source == MODEL_NAME:
    last_checkpoint = get_last_checkpoint('train_result') if Path('train_result').exists() else None
    trainer.train(resume_from_checkpoint=last_checkpoint)
    trainer.save_model(SAVED_MODEL)
    tokenizer.save_pretrained(SAVED_MODEL)
else:
    print('Загружена ранее обученная модель:', SAVED_MODEL.resolve())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2268.20it/s]

Загружена ранее обученная модель: C:\Users\Admin\programms\mei-czti\lab-1\res_model_base


## 8. Проверка модели на тестовой выборке

`test` не использовался ни для обучения, ни для выбора лучшей эпохи. Поэтому этот отчёт показывает итоговое качество распознавания имён и фамилий.

In [8]:
test_prediction = trainer.predict(datasets['test'])
ner_true, ner_predicted = sequences_from_predictions(
    test_prediction.predictions, test_prediction.label_ids
)

print(classification_report(
    ner_true, ner_predicted, scheme=IOB2, digits=4, zero_division=0
))
ner_report = classification_report(
    ner_true,
    ner_predicted,
    scheme=IOB2,
    output_dict=True,
    zero_division=0,
)

C:\Users\Admin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


              precision    recall  f1-score   support

        NAME     0.8876    0.9186    0.9029        86
     SURNAME     0.8243    0.9385    0.8777       130

   micro avg     0.8481    0.9306    0.8874       216
   macro avg     0.8560    0.9285    0.8903       216
weighted avg     0.8495    0.9306    0.8877       216



## 9. Структурированные персональные данные

Для данных со строгим форматом нейросеть не нужна. Регулярные выражения находят кандидатов, а дополнительные функции проверяют:

- ИНН — официальные весовые коэффициенты;
- банковскую карту — алгоритм Луна;
- СНИЛС — взвешенную сумму первых девяти цифр.

In [9]:
import re

EMAIL_RE = re.compile(r'(?<![\w.+-])[\w.%+-]+@[\w-]+(?:\.[\w-]+)+(?![\w.-])', re.I)
PHONE_RE = re.compile(r'(?<!\d)(?:\+7|8)[ \-]*\(?\d{3}\)?[ \-]*\d{3}[ \-]*\d{2}[ \-]*\d{2}(?!\d)')
CARD_RE = re.compile(r'(?<!\d)(?:\d[ \-]?){15}\d(?!\d)')
INN12_RE = re.compile(r'(?<!\d)(?:\d[ \-]?){11}\d(?!\d)')
INN10_RE = re.compile(r'(?<!\d)(?:\d[ \-]?){9}\d(?!\d)')
SNILS_RE = re.compile(r'(?<!\d)\d{3}[ \-]\d{3}[ \-]\d{3} ?\d{2}(?!\d)')
PASSPORT_RE = re.compile(
    r'(?:паспорт(?:а|у|ом)?|серия и номер паспорта)\s*[:№-]?\s*'
    r'(?P<value>\d{2} ?\d{2} \d{6})(?!\d)',
    re.I,
)

def only_digits(value):
    return ''.join(character for character in value if character.isdigit())

def valid_inn(value):
    digits = only_digits(value)
    if len(digits) == 10:
        weights = (2, 4, 10, 3, 5, 9, 4, 6, 8)
        check = sum(int(digits[i]) * weights[i] for i in range(9)) % 11 % 10
        return check == int(digits[9])
    if len(digits) == 12:
        weights_11 = (7, 2, 4, 10, 3, 5, 9, 4, 6, 8)
        weights_12 = (3, 7, 2, 4, 10, 3, 5, 9, 4, 6, 8)
        check_11 = sum(int(digits[i]) * weights_11[i] for i in range(10)) % 11 % 10
        check_12 = sum(int(digits[i]) * weights_12[i] for i in range(11)) % 11 % 10
        return check_11 == int(digits[10]) and check_12 == int(digits[11])
    return False

def valid_card(value):
    digits = only_digits(value)
    if len(digits) != 16 or len(set(digits)) == 1:
        return False
    total = 0
    for index, character in enumerate(digits):
        digit = int(character)
        if index % 2 == 0:
            digit *= 2
            if digit > 9:
                digit -= 9
        total += digit
    return total % 10 == 0

def valid_snils(value):
    digits = only_digits(value)
    if len(digits) != 11 or int(digits[:9]) <= 1001998:
        return False
    total = sum(int(digits[i]) * (9 - i) for i in range(9))
    check = total if total < 100 else total % 101
    if check == 100:
        check = 0
    return check == int(digits[9:])

def overlaps(left, right):
    return left[1] < right[2] and right[1] < left[2]

def find_structured(text):
    candidates = []
    for match in EMAIL_RE.finditer(text):
        candidates.append((100, ('EMAIL', match.start(), match.end())))
    for match in CARD_RE.finditer(text):
        if valid_card(match.group()):
            candidates.append((95, ('BANK_CARD', match.start(), match.end())))
    for match in SNILS_RE.finditer(text):
        if valid_snils(match.group()):
            candidates.append((95, ('SNILS', match.start(), match.end())))
    for match in PASSPORT_RE.finditer(text):
        start, end = match.span('value')
        candidates.append((90, ('PASSPORT', start, end)))
    for pattern in (INN12_RE, INN10_RE):
        for match in pattern.finditer(text):
            if valid_inn(match.group()):
                candidates.append((85, ('INN', match.start(), match.end())))
    for match in PHONE_RE.finditer(text):
        candidates.append((80, ('PHONE', match.start(), match.end())))

    selected = []
    for priority, candidate in sorted(candidates, key=lambda item: (-item[0], item[1][1])):
        if not any(overlaps(candidate, current) for current in selected):
            selected.append(candidate)
    return sorted(selected, key=lambda entity: entity[1])

## 10. Объединение RuBERT и регулярных выражений

Модель возвращает символьные границы имён и фамилий. Затем к ним добавляются структурированные атрибуты. Пересекающиеся сущности отбрасываются, поэтому каждый токен относится только к одному классу.

In [10]:
from transformers import pipeline

person_ner = pipeline(
    'token-classification',
    model=trainer.model,
    tokenizer=tokenizer,
    aggregation_strategy='first',
    device=0 if torch.cuda.is_available() else -1,
)

def recognize(text):
    entities = find_structured(text)
    for item in person_ner(text):
        label = item.get('entity_group', item.get('entity', ''))
        label = re.sub(r'^[BI]-', '', label)
        candidate = (label, int(item['start']), int(item['end']))
        if label in TARGET_TYPES and not any(overlaps(candidate, current) for current in entities):
            entities.append(candidate)
    return [list(entity) for entity in sorted(entities, key=lambda entity: entity[1])]

TOKEN_RE = re.compile(r'\w+|[^\w\s]', re.UNICODE)

def make_bio_markup(text, entities):
    result = []
    previous_entity = None
    for match in TOKEN_RE.finditer(text):
        current = next((
            tuple(entity) for entity in entities
            if match.start() < entity[2] and entity[1] < match.end()
        ), None)
        if current is None:
            tag = 'O'
        else:
            tag = ('I-' if current == previous_entity else 'B-') + current[0]
        result.append((match.group(), tag, match.start(), match.end()))
        previous_entity = current
    return result

## 11. Не менее 50 тестов на каждый атрибут

Для `NAME` и `SURNAME` используются реальные сущности из тестовой части factRuEval. Для каждого структурированного атрибута Faker создаёт 60 независимых примеров. Затем считается F1 каждого класса и средневзвешенная F1 всей программы.

In [11]:
from collections import defaultdict
from faker import Faker
from random import Random

fake = Faker('ru_RU')
Faker.seed(42)
random_generator = Random(42)

def make_inn(length):
    if length == 10:
        prefix = ''.join(str(random_generator.randrange(10)) for _ in range(9))
        weights = (2, 4, 10, 3, 5, 9, 4, 6, 8)
        check = sum(int(prefix[i]) * weights[i] for i in range(9)) % 11 % 10
        return prefix + str(check)
    prefix = ''.join(str(random_generator.randrange(10)) for _ in range(10))
    w11 = (7, 2, 4, 10, 3, 5, 9, 4, 6, 8)
    c11 = sum(int(prefix[i]) * w11[i] for i in range(10)) % 11 % 10
    first_11 = prefix + str(c11)
    w12 = (3, 7, 2, 4, 10, 3, 5, 9, 4, 6, 8)
    c12 = sum(int(first_11[i]) * w12[i] for i in range(11)) % 11 % 10
    return first_11 + str(c12)

def make_card():
    prefix = '4' + ''.join(str(random_generator.randrange(10)) for _ in range(14))
    for check in range(10):
        candidate = prefix + str(check)
        if valid_card(candidate):
            return ' '.join(candidate[i:i + 4] for i in range(0, 16, 4))

def make_snils():
    number = f'{random_generator.randrange(1001999, 1000000000):09d}'
    total = sum(int(number[i]) * (9 - i) for i in range(9))
    check = total if total < 100 else total % 101
    if check == 100:
        check = 0
    value = number + f'{check:02d}'
    return f'{value[:3]}-{value[3:6]}-{value[6:9]} {value[9:]}'

def add_test(tests, label, text, value):
    start = text.index(value)
    tests.append((text, (label, start, start + len(value))))

structured_tests = []
for index in range(60):
    email = fake.ascii_free_email()
    add_test(structured_tests, 'EMAIL', f'электронная почта: {email}', email)

    inn = make_inn(10 if index % 2 == 0 else 12)
    add_test(structured_tests, 'INN', f'ИНН: {inn}', inn)

    digits = '9' + ''.join(str(random_generator.randrange(10)) for _ in range(9))
    phone = f'+7 ({digits[:3]}) {digits[3:6]}-{digits[6:8]}-{digits[8:]}'
    add_test(structured_tests, 'PHONE', f'телефон: {phone}', phone)

    card = make_card()
    add_test(structured_tests, 'BANK_CARD', f'банковская карта: {card}', card)

    snils = make_snils()
    add_test(structured_tests, 'SNILS', f'СНИЛС: {snils}', snils)

    passport = f'{random_generator.randrange(10, 100):02d} {random_generator.randrange(10, 100):02d} {random_generator.randrange(1000000):06d}'
    add_test(structured_tests, 'PASSPORT', f'паспорт: {passport}', passport)

structured_tests += [
    ('неверный ИНН: 7707083894', None),
    ('неверная карта: 4111 1111 1111 1112', None),
    ('неверный СНИЛС: 112-233-445 96', None),
]

tp, fp, fn = Counter(), Counter(), Counter()
for text, gold in structured_tests:
    gold_set = set() if gold is None else {gold}
    predicted_set = set(find_structured(text))
    tp.update(label for label, start, end in gold_set & predicted_set)
    fp.update(label for label, start, end in predicted_set - gold_set)
    fn.update(label for label, start, end in gold_set - predicted_set)

all_metrics = {}
for label in ('NAME', 'SURNAME'):
    row = ner_report[label]
    all_metrics[label] = {
        'precision': row['precision'],
        'recall': row['recall'],
        'f1': row['f1-score'],
        'support': int(row['support']),
    }

for label in ('EMAIL', 'INN', 'PHONE', 'BANK_CARD', 'SNILS', 'PASSPORT'):
    precision = tp[label] / (tp[label] + fp[label]) if tp[label] + fp[label] else 0
    recall = tp[label] / (tp[label] + fn[label]) if tp[label] + fn[label] else 0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0
    all_metrics[label] = {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'support': tp[label] + fn[label],
    }

print(f"{'Атрибут':<14} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Тестов':>10}")
print('-' * 58)
for label, row in all_metrics.items():
    print(f"{label:<14} {row['precision']:>10.3f} {row['recall']:>10.3f} {row['f1']:>10.3f} {row['support']:>10}")

total_support = sum(row['support'] for row in all_metrics.values())
weighted_f1 = sum(row['f1'] * row['support'] for row in all_metrics.values()) / total_support
print('-' * 58)
print(f'Средневзвешенная F1: {weighted_f1:.3f}')

assert all(row['support'] >= 50 for row in all_metrics.values()), 'Недостаточно тестов'
assert weighted_f1 >= 0.85, 'F1 ниже требуемых 0.85'

Атрибут         Precision     Recall         F1     Тестов
----------------------------------------------------------
NAME                0.888      0.919      0.903         86
SURNAME             0.824      0.938      0.878        130
EMAIL               1.000      1.000      1.000         60
INN                 1.000      1.000      1.000         60
PHONE               1.000      1.000      1.000         60
BANK_CARD           1.000      1.000      1.000         60
SNILS               1.000      1.000      1.000         60
PASSPORT            1.000      1.000      1.000         60
----------------------------------------------------------
Средневзвешенная F1: 0.958


## 12. Демонстрация программы

Функция `recognize(text)` возвращает требуемый список `[название атрибута, start, end]`. Правая граница `end` не включается, поэтому исходное значение можно получить как `text[start:end]`.

In [12]:
text = (
    'Борис Громов, ИНН 7707083893, телефон +7 (999) 123-45-67, '
    'e-mail ivanov@example.ru, карта 4111 1111 1111 1111, '
    'СНИЛС 112-233-445 95, паспорт 45 08 123456.'
)

entities = recognize(text)
print(text)
print('\nНайденные сущности:')
for entity in entities:
    print(entity, '=>', repr(text[entity[1]:entity[2]]))

print('\nBIO-разметка:')
for token, tag, start, end in make_bio_markup(text, entities):
    print(f'{token!r:<16} {tag:<16} [{start:>3}, {end:>3})')

Борис Громов, ИНН 7707083893, телефон +7 (999) 123-45-67, e-mail ivanov@example.ru, карта 4111 1111 1111 1111, СНИЛС 112-233-445 95, паспорт 45 08 123456.

Найденные сущности:
['NAME', 0, 5] => 'Борис'
['SURNAME', 6, 12] => 'Громов'
['INN', 18, 28] => '7707083893'
['PHONE', 38, 56] => '+7 (999) 123-45-67'
['EMAIL', 65, 82] => 'ivanov@example.ru'
['BANK_CARD', 90, 109] => '4111 1111 1111 1111'
['SNILS', 117, 131] => '112-233-445 95'
['PASSPORT', 141, 153] => '45 08 123456'

BIO-разметка:
'Борис'          B-NAME           [  0,   5)
'Громов'         B-SURNAME        [  6,  12)
','              O                [ 12,  13)
'ИНН'            O                [ 14,  17)
'7707083893'     B-INN            [ 18,  28)
','              O                [ 28,  29)
'телефон'        O                [ 30,  37)
'+'              B-PHONE          [ 38,  39)
'7'              I-PHONE          [ 39,  40)
'('              I-PHONE          [ 41,  42)
'999'            I-PHONE          [ 42,  45)
')'          

## Кратко для защиты

1. factRuEval содержит тексты и символьную разметку сущностей. Из него используются `name` и `surname`.
2. RuBERT получает текст, разбивает его на WordPiece-субтокены и определяет BIO-метку каждого первого субтокена слова.
3. `-100` нужен, чтобы не считать ошибку на служебных и повторных субтокенах.
4. Регулярные выражения используются для данных со строгим форматом.
5. Контрольные разряды защищают от распознавания случайных наборов цифр как ИНН, карту или СНИЛС.
6. Precision показывает точность найденных сущностей, recall — полноту, F1 объединяет обе метрики.
7. Каждый токен получает ровно один BIO-тег, а итоговые символьные индексы нужны для последующего обезличивания.